# Post-Processing & Span Smoothing (`BertCRF`)

Menggunakan checkpoint terbaik `outputs/task_3/runs/baseline/bert_crf.pt` pada `tsd_trial` (`tsd_test` tetap tidak disentuh) untuk menguji:
1. **Whole-Word Subword Expansion + `fix_spans()`**
2. **Sentence-Level Confidence Gate (`tau`)** untuk memperbaiki `Empty F1` (`0,02326` pada baseline mentah)
3. **Long-Span Capping Ablation** (inspeksi dampak pembatasan span panjang)

In [1]:
import json
from pathlib import Path
import sys

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchcrf import CRF
from transformers import AutoConfig, AutoModel, AutoTokenizer

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src" / "task_3").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.task_3.config import DEFAULT_BACKBONE, EVAL_DIR, ID2LABEL, MAX_LEN, NUM_TAGS, RESULTS_DIR, RUNS_DIR, SEEDS, set_seed
from src.task_3.dataset import ToxicSpanCollator, ToxicSpanDataset, bio_to_char_spans
from src.task_3.metrics import evaluate_bio_tags, evaluate_char_spans

if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))
from fix_spans import _contiguous_ranges, fix_spans


def log(tag, message, *, err=None):
    if err is None:
        err = tag in ("ERROR", "FATAL", "WARN")
    print(f"[{tag}] {message}", file=sys.stderr if err else sys.stdout, flush=True)


def print_table(title, rows, line_width=49):
    print(title)
    widths = [max(len(str(c)) for c in col) for col in zip(*rows)]
    total = sum(widths) + 3 * (len(widths) - 1)
    print("-" * max(line_width, total))
    for row in rows:
        print(" | ".join(str(c).ljust(w) for c, w in zip(row, widths)).rstrip())


class BertCRF(nn.Module):
    def __init__(self, model_name, num_tags=NUM_TAGS, dropout=0.2):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, num_tags)
        self.crf = CRF(num_tags=num_tags, batch_first=True)

    def decode_with_emissions(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask.long())
        emissions = self.classifier(self.dropout(outputs.last_hidden_state))
        paths = self.crf.decode(emissions, mask=attention_mask)
        return paths, emissions


set_seed(SEEDS[0])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_PATH = RUNS_DIR / "baseline" / "bert_crf.pt"
BASE_SUMMARY_PATH = RESULTS_DIR / "baseline" / "baseline_summary.json"
OUT_POST = RESULTS_DIR / "postprocess"
OUT_POST.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_BACKBONE, use_fast=True)
trial_ds = ToxicSpanDataset("trial", tokenizer, max_length=MAX_LEN)
collator = ToxicSpanCollator(pad_token_id=tokenizer.pad_token_id)
trial_loader = DataLoader(trial_ds, batch_size=32, shuffle=False, collate_fn=collator)

model = BertCRF(DEFAULT_BACKBONE).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()

raw_pred_seqs = []
toxic_probs_per_row = []
with torch.no_grad():
    for batch in trial_loader:
        paths, emissions = model.decode_with_emissions(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE),
        )
        probs = torch.softmax(emissions, dim=-1).cpu()
        for row_idx, path in enumerate(paths):
            seq_len = len(path)
            raw_pred_seqs.append(path)
            p_tox = (1.0 - probs[row_idx, :seq_len, 0]).tolist()
            toxic_probs_per_row.append(p_tox)

word_ids_per_row = [
    tokenizer(r["text"], truncation=True, max_length=MAX_LEN).word_ids(0)
    for r in trial_ds.rows
]
trial_golds = [r["spans"] for r in trial_ds.rows]
gold_tags = [[ID2LABEL[t] for t in s["labels"]] for s in trial_ds.samples]

log("START", f"Loaded checkpoint {CKPT_PATH.name} and cached {len(raw_pred_seqs)} trial emissions on {DEVICE}")

W1121 16:06:21.290000 17056 venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[START] Loaded checkpoint bert_crf.pt and cached 690 trial emissions on cuda


## 1. Eksperimen 1 — Whole-Word Expansion & `fix_spans()`

In [2]:
def expand_whole_words(pred_seq, word_ids):
    toxic_word_ids = {
        wid
        for tag_id, wid in zip(pred_seq, word_ids)
        if wid is not None and tag_id != 0
    }
    if not toxic_word_ids:
        return list(pred_seq)

    out = list(pred_seq)
    for idx, wid in enumerate(word_ids):
        if wid in toxic_word_ids:
            if out[idx] == 0:
                prev_toxic = idx > 0 and out[idx - 1] != 0
                out[idx] = 2 if prev_toxic else 1
            elif idx > 0 and word_ids[idx - 1] == wid and out[idx - 1] != 0:
                out[idx] = 2
    return out


def eval_variant(seqs, use_fix_spans=False):
    spans = [
        bio_to_char_spans(seq, s["offset_mapping"], r["text"])
        for seq, s, r in zip(seqs, trial_ds.samples, trial_ds.rows)
    ]
    if use_fix_spans:
        spans = [fix_spans(sp, r["text"]) for sp, r in zip(spans, trial_ds.rows)]
    char_stat = evaluate_char_spans(spans, trial_golds)
    pred_tags = [[ID2LABEL[t] for t in seq] for seq in seqs]
    seq_stat = evaluate_bio_tags(pred_tags, gold_tags)
    return char_stat, seq_stat, spans


ww_seqs = [expand_whole_words(seq, wids) for seq, wids in zip(raw_pred_seqs, word_ids_per_row)]

variants = [
    ("BERT-CRF (Raw)", raw_pred_seqs, False),
    ("+ Whole-Word Expansion", ww_seqs, False),
    ("+ fix_spans()", raw_pred_seqs, True),
    ("+ Whole-Word + fix_spans()", ww_seqs, True),
]

exp1_rows = [["Varian", "Char F1", "SEM", "Empty F1", "NonEmpty F1", "Seqeval F1"]]
for name, seqs, do_fix in variants:
    c_stat, s_stat, _ = eval_variant(seqs, use_fix_spans=do_fix)
    exp1_rows.append([
        name,
        f"{c_stat['f1']:.5f}",
        f"{c_stat['sem']:.5f}",
        f"{c_stat['empty_f1']:.5f}",
        f"{c_stat['non_empty_f1']:.5f}",
        f"{s_stat['f1']:.5f}",
    ])

log("CHECKPOINT 1", "Eksperimen Whole-Word Expansion & fix_spans selesai")
print_table("Eksperimen 1: Whole-Word & fix_spans (tsd_trial)", exp1_rows)

[CHECKPOINT 1] Eksperimen Whole-Word Expansion & fix_spans selesai
Eksperimen 1: Whole-Word & fix_spans (tsd_trial)
------------------------------------------------------------------------------------
Varian                     | Char F1 | SEM     | Empty F1 | NonEmpty F1 | Seqeval F1
BERT-CRF (Raw)             | 0.68038 | 0.01370 | 0.02326  | 0.72405     | 0.58598
+ Whole-Word Expansion     | 0.68051 | 0.01371 | 0.02326  | 0.72419     | 0.58708
+ fix_spans()              | 0.68038 | 0.01370 | 0.02326  | 0.72405     | 0.58598
+ Whole-Word + fix_spans() | 0.68051 | 0.01371 | 0.02326  | 0.72419     | 0.58708


## 2. Eksperimen 2 — Confidence Gate (`Empty F1` Mitigation)

In [3]:
def apply_empty_gate(seqs, p_tox_list, tau, use_fix_spans=True):
    gated_seqs = []
    for seq, p_tox in zip(seqs, p_tox_list):
        tox_scores = [p for tag_id, p in zip(seq, p_tox) if tag_id != 0]
        if tox_scores and max(tox_scores) < tau:
            gated_seqs.append([0] * len(seq))
        else:
            gated_seqs.append(seq)
    c_stat, s_stat, spans = eval_variant(gated_seqs, use_fix_spans=use_fix_spans)
    n_empty_pred = sum(len(sp) == 0 for sp in spans)
    return c_stat, s_stat, spans, gated_seqs, n_empty_pred


taus = [0.0, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.85, 0.90]
exp2_rows = [["tau (max P_tox)", "Char F1", "Empty F1", "NonEmpty F1", "Seqeval F1", "Pred Empty%"]]
gate_results = []

for tau in taus:
    c_stat, s_stat, spans, gated_seqs, n_empty = apply_empty_gate(ww_seqs, toxic_probs_per_row, tau, use_fix_spans=True)
    gate_results.append((tau, c_stat, s_stat, spans, gated_seqs))
    exp2_rows.append([
        f"{tau:.2f}",
        f"{c_stat['f1']:.5f}",
        f"{c_stat['empty_f1']:.5f}",
        f"{c_stat['non_empty_f1']:.5f}",
        f"{s_stat['f1']:.5f}",
        f"{100.0 * n_empty / len(trial_ds):.2f}%",
    ])

best_tau, best_c, best_s, best_spans, best_seqs = max(gate_results, key=lambda x: x[1]["f1"])
log("CHECKPOINT 2", f"Best tau={best_tau:.2f} -> Char F1={best_c['f1']:.5f}, Empty F1={best_c['empty_f1']:.5f}")
print_table("Eksperimen 2: Sweep Confidence Gate pada +Whole-Word +fix_spans (tsd_trial)", exp2_rows)

[CHECKPOINT 2] Best tau=0.00 -> Char F1=0.68051, Empty F1=0.02326
Eksperimen 2: Sweep Confidence Gate pada +Whole-Word +fix_spans (tsd_trial)
-----------------------------------------------------------------------------
tau (max P_tox) | Char F1 | Empty F1 | NonEmpty F1 | Seqeval F1 | Pred Empty%
0.00            | 0.68051 | 0.02326  | 0.72419     | 0.58708    | 1.30%
0.30            | 0.68051 | 0.02326  | 0.72419     | 0.58708    | 1.30%
0.40            | 0.68051 | 0.02326  | 0.72419     | 0.58708    | 1.30%
0.50            | 0.68051 | 0.02326  | 0.72419     | 0.58708    | 1.30%
0.60            | 0.68051 | 0.02326  | 0.72419     | 0.58708    | 1.30%
0.70            | 0.68051 | 0.02326  | 0.72419     | 0.58740    | 1.45%
0.80            | 0.67387 | 0.04651  | 0.71556     | 0.58882    | 3.33%
0.85            | 0.66470 | 0.09302  | 0.70269     | 0.58923    | 5.94%
0.90            | 0.64087 | 0.25581  | 0.66646     | 0.58685    | 13.19%


## 3. Eksperimen 3 — Long-Span Capping Ablation

In [4]:
def cap_long_spans(spans, text, max_words):
    if max_words is None or not spans:
        return spans
    kept = []
    for begin, end in _contiguous_ranges(spans):
        n_words = len(text[begin : end + 1].split())
        if n_words <= max_words:
            kept.extend(range(begin, end + 1))
    return kept


exp3_rows = [["Max Words/Span", "Char F1", "Empty F1", "NonEmpty F1", "Spans Dropped"]]
for max_w in [None, 12, 8, 6, 4]:
    capped = [cap_long_spans(sp, r["text"], max_w) for sp, r in zip(best_spans, trial_ds.rows)]
    dropped = sum(
        len(_contiguous_ranges(sp_orig)) - len(_contiguous_ranges(sp_cap))
        for sp_orig, sp_cap in zip(best_spans, capped)
    )
    c_stat = evaluate_char_spans(capped, trial_golds)
    exp3_rows.append([
        "No cap" if max_w is None else str(max_w),
        f"{c_stat['f1']:.5f}",
        f"{c_stat['empty_f1']:.5f}",
        f"{c_stat['non_empty_f1']:.5f}",
        str(dropped),
    ])

log("CHECKPOINT 3", "Eksperimen Long-Span Capping selesai")
print_table("Eksperimen 3: Long-Span Capping di atas Best Post-Processing", exp3_rows)

[CHECKPOINT 3] Eksperimen Long-Span Capping selesai
Eksperimen 3: Long-Span Capping di atas Best Post-Processing
-----------------------------------------------------------------
Max Words/Span | Char F1 | Empty F1 | NonEmpty F1 | Spans Dropped
No cap         | 0.68051 | 0.02326  | 0.72419     | 0
12             | 0.67982 | 0.02326  | 0.72345     | 6
8              | 0.67994 | 0.02326  | 0.72358     | 11
6              | 0.67453 | 0.02326  | 0.71782     | 24
4              | 0.66693 | 0.06977  | 0.70662     | 47


## 4. Ringkasan Perbandingan Akhir & Simpan Konfigurasi

In [5]:
base_summary = json.loads(BASE_SUMMARY_PATH.read_text(encoding="utf-8"))
raw_c, raw_s, _ = eval_variant(raw_pred_seqs, use_fix_spans=False)
ww_fix_c, ww_fix_s, _ = eval_variant(ww_seqs, use_fix_spans=True)

final_rows = [
    ["Pipeline", "Char F1", "SEM", "Empty F1", "NonEmpty F1", "Seqeval F1"],
    [
        "Lexicon Baseline",
        f"{base_summary['lexicon']['char']['f1']:.5f}",
        f"{base_summary['lexicon']['char']['sem']:.5f}",
        f"{base_summary['lexicon']['char']['empty_f1']:.5f}",
        f"{base_summary['lexicon']['char']['non_empty_f1']:.5f}",
        "-",
    ],
    [
        "BERT-Linear (Raw)",
        f"{base_summary['bert_linear']['char']['f1']:.5f}",
        f"{base_summary['bert_linear']['char']['sem']:.5f}",
        f"{base_summary['bert_linear']['char']['empty_f1']:.5f}",
        f"{base_summary['bert_linear']['char']['non_empty_f1']:.5f}",
        f"{base_summary['bert_linear']['seqeval']['f1']:.5f}",
    ],
    [
        "BERT-CRF (Raw)",
        f"{raw_c['f1']:.5f}",
        f"{raw_c['sem']:.5f}",
        f"{raw_c['empty_f1']:.5f}",
        f"{raw_c['non_empty_f1']:.5f}",
        f"{raw_s['f1']:.5f}",
    ],
    [
        "BERT-CRF + WW + fix_spans",
        f"{ww_fix_c['f1']:.5f}",
        f"{ww_fix_c['sem']:.5f}",
        f"{ww_fix_c['empty_f1']:.5f}",
        f"{ww_fix_c['non_empty_f1']:.5f}",
        f"{ww_fix_s['f1']:.5f}",
    ],
    [
        f"BERT-CRF + WW + fix_spans + Gate(tau={best_tau:.2f})",
        f"{best_c['f1']:.5f}",
        f"{best_c['sem']:.5f}",
        f"{best_c['empty_f1']:.5f}",
        f"{best_c['non_empty_f1']:.5f}",
        f"{best_s['f1']:.5f}",
    ],
]
print_table("Ringkasan Ablation Phase 4 (tsd_trial)", final_rows)

post_payload = {
    "best_tau": best_tau,
    "raw_crf": {"char": raw_c, "seqeval": raw_s},
    "ww_fix_spans": {"char": ww_fix_c, "seqeval": ww_fix_s},
    "ww_fix_spans_gate": {"char": best_c, "seqeval": best_s},
}
(OUT_POST / "postprocess_summary.json").write_text(json.dumps(post_payload, indent=2), encoding="utf-8")
(OUT_POST / "bert_crf_post_trial_preds.json").write_text(json.dumps(best_spans), encoding="utf-8")
log("DONE", f"Hasil post-processing tersimpan di {OUT_POST}")

Ringkasan Ablation Phase 4 (tsd_trial)
----------------------------------------------------------------------------------------------------
Pipeline                                   | Char F1 | SEM     | Empty F1 | NonEmpty F1 | Seqeval F1
Lexicon Baseline                           | 0.59794 | 0.01545 | 0.09302  | 0.63150     | -
BERT-Linear (Raw)                          | 0.67110 | 0.01369 | 0.04651  | 0.71261     | 0.54688
BERT-CRF (Raw)                             | 0.68038 | 0.01370 | 0.02326  | 0.72405     | 0.58598
BERT-CRF + WW + fix_spans                  | 0.68051 | 0.01371 | 0.02326  | 0.72419     | 0.58708
BERT-CRF + WW + fix_spans + Gate(tau=0.00) | 0.68051 | 0.01371 | 0.02326  | 0.72419     | 0.58708
[DONE] Hasil post-processing tersimpan di C:\Users\ASUS\Desktop\nlp\tubes\IF5153-Chat-Moderation\outputs\task_3\results\postprocess
